In [3]:
# ============================================================
# 🇳🇱 HOLLAND AGRICULTURAL MACHINERY
# 🚜 INTERACTIVE HTML DASHBOARD - FIXED VERSION
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

import sys
import subprocess

packages = ["pandas", "numpy"]

for package in packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", package, "--quiet"
    ])

print("✅ Required Python libraries installed!")


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import os
import json
import webbrowser
from pathlib import Path


# ============================================================
# 3. FILE SETTINGS
# ============================================================

CSV_FILE = r"C:\Users\Nasar\Documents\DATAANALYSIS\holland_tractor_showroom_100_records.csv"

OUTPUT_DIR = r"C:\Users\Nasar\Documents\DATAANALYSIS\tractor_analysis_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\n" + "=" * 70)
print("🇳🇱 HOLLAND AGRICULTURAL MACHINERY")
print("=" * 70)

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(
        f"""
❌ CSV FILE NOT FOUND!

Please check this path:

{CSV_FILE}
"""
    )

df = pd.read_csv(CSV_FILE)

print("✅ Dataset loaded successfully!")
print("Records:", len(df))


# ============================================================
# 5. CLEAN DATA
# ============================================================

# Clean column names
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

# Clean text columns
for col in df.select_dtypes(include="object").columns:

    df[col] = (
        df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

# Clean numeric columns
for col in ["Price_EUR", "Working_Hours"]:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].fillna(
            df[col].median()
        )


# ============================================================
# 6. CHECK COLUMNS
# ============================================================

required_columns = [
    "Price_EUR",
    "Working_Hours",
    "Status",
    "Category",
    "Brand"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:

    raise ValueError(
        f"""
❌ These columns are missing:

{missing}

Your CSV columns are:

{df.columns.tolist()}
"""
    )


print("✅ All required columns found!")


# ============================================================
# 7. NORMALIZE FILTER COLUMNS
# ============================================================
# This fixes the 0-record filtering problem.
# Spaces and capital letters will not matter.

df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
)

df["Brand"] = (
    df["Brand"]
    .astype(str)
    .str.strip()
)

df["Status"] = (
    df["Status"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 8. REMOVE DUPLICATES
# ============================================================

df.drop_duplicates(
    inplace=True
)


# ============================================================
# 9. BUSINESS CALCULATIONS
# ============================================================

total_machines = len(df)

inventory_value = df["Price_EUR"].sum()

average_price = df["Price_EUR"].mean()

average_hours = df["Working_Hours"].mean()

in_stock = (
    df["Status"]
    .str.lower()
    .eq("in stock")
    .sum()
)

sold = (
    df["Status"]
    .str.lower()
    .eq("sold")
    .sum()
)

reserved = (
    df["Status"]
    .str.lower()
    .eq("reserved")
    .sum()
)

available_soon = (
    df["Status"]
    .str.lower()
    .eq("available soon")
    .sum()
)


# ============================================================
# 10. SAVE CLEAN DATA
# ============================================================

cleaned_file = os.path.join(
    OUTPUT_DIR,
    "cleaned_holland_machinery_data.csv"
)

df.to_csv(
    cleaned_file,
    index=False
)


# ============================================================
# 11. CONVERT DATA TO JSON
# ============================================================

data_json = json.dumps(
    df.to_dict(orient="records"),
    ensure_ascii=False
)


# ============================================================
# 12. HTML DASHBOARD
# ============================================================

html = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta name="viewport"
content="width=device-width, initial-scale=1.0">

<title>
Holland Agricultural Machinery Dashboard
</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>


<style>

/* =========================================================
   GENERAL
   ========================================================= */

* {{
    box-sizing: border-box;
}}

body {{

    margin: 0;

    font-family:
    Arial,
    Helvetica,
    sans-serif;

    background: #F3F7F3;

    color: #17221D;

}}

.container {{

    width: 94%;

    max-width: 1500px;

    margin: auto;

    padding-bottom: 50px;

}}


/* =========================================================
   HEADER
   ========================================================= */

.header {{

    margin-top: 25px;

    padding: 35px;

    border-radius: 22px;

    color: white;

    background:
    linear-gradient(
        135deg,
        #12372A,
        #2E8B57
    );

    box-shadow:
    0 8px 25px
    rgba(18,55,42,0.18);

}}

.header h1 {{

    margin: 0;

    font-size: 34px;

}}

.header p {{

    margin: 10px 0 0;

    font-size: 16px;

    opacity: .9;

}}


/* =========================================================
   FILTERS
   ========================================================= */

.filters {{

    background: white;

    margin-top: 20px;

    padding: 22px;

    border-radius: 18px;

    display: grid;

    grid-template-columns:
    repeat(4,1fr);

    gap: 15px;

    box-shadow:
    0 5px 20px
    rgba(0,0,0,.06);

}}

.filter label {{

    display: block;

    font-weight: bold;

    color: #12372A;

    margin-bottom: 7px;

}}

select,
input {{

    width: 100%;

    padding: 12px;

    border-radius: 10px;

    border:
    1px solid #D7E0DA;

    background: #FAFCFA;

    font-size: 14px;

}}

select:focus,
input:focus {{

    outline: none;

    border-color: #2E8B57;

}}


/* =========================================================
   RESET BUTTON
   ========================================================= */

.reset-area {{

    text-align: right;

    margin-top: 12px;

}}

.reset-btn {{

    border: none;

    padding: 12px 22px;

    border-radius: 10px;

    background: #C76D4F;

    color: white;

    font-weight: bold;

    cursor: pointer;

}}

.reset-btn:hover {{

    opacity: .85;

}}


/* =========================================================
   KPI
   ========================================================= */

.kpis {{

    display: grid;

    grid-template-columns:
    repeat(5,1fr);

    gap: 16px;

    margin-top: 20px;

}}

.kpi {{

    background: white;

    padding: 22px;

    border-radius: 18px;

    border-top:
    5px solid #2E8B57;

    box-shadow:
    0 5px 18px
    rgba(0,0,0,.06);

    transition: .2s;

}}

.kpi:hover {{

    transform:
    translateY(-4px);

}}

.kpi-title {{

    font-size: 12px;

    font-weight: bold;

    color: #4F8A7A;

}}

.kpi-value {{

    font-size: 26px;

    font-weight: bold;

    margin-top: 8px;

    color: #12372A;

}}


/* =========================================================
   CHARTS
   ========================================================= */

.chart-grid {{

    display: grid;

    grid-template-columns:
    1fr 1fr;

    gap: 20px;

    margin-top: 20px;

}}

.chart-card {{

    background: white;

    border-radius: 18px;

    padding: 10px;

    box-shadow:
    0 5px 18px
    rgba(0,0,0,.06);

}}


/* =========================================================
   TABLE
   ========================================================= */

.table-card {{

    background: white;

    margin-top: 20px;

    padding: 22px;

    border-radius: 18px;

    box-shadow:
    0 5px 18px
    rgba(0,0,0,.06);

}}

.table-title {{

    font-size: 20px;

    font-weight: bold;

    color: #12372A;

    margin-bottom: 15px;

}}

.table-wrapper {{

    overflow-x: auto;

    max-height: 500px;

}}

table {{

    width: 100%;

    border-collapse: collapse;

}}

th {{

    background: #12372A;

    color: white;

    padding: 12px;

    text-align: left;

    position: sticky;

    top: 0;

}}

td {{

    padding: 11px;

    border-bottom:
    1px solid #E7ECE8;

}}

tr:hover {{

    background: #F1F7F3;

}}


/* =========================================================
   NO RESULTS
   ========================================================= */

.no-results {{

    display: none;

    text-align: center;

    padding: 30px;

    color: #C76D4F;

    font-size: 18px;

    font-weight: bold;

}}


/* =========================================================
   FOOTER
   ========================================================= */

.footer {{

    text-align: center;

    margin-top: 30px;

    padding: 20px;

    color: #4F8A7A;

    font-size: 13px;

}}


/* =========================================================
   RESPONSIVE
   ========================================================= */

@media(max-width:1000px) {{

    .kpis {{

        grid-template-columns:
        repeat(2,1fr);

    }}

    .filters {{

        grid-template-columns:
        repeat(2,1fr);

    }}

    .chart-grid {{

        grid-template-columns:
        1fr;

    }}

}}

@media(max-width:600px) {{

    .kpis,
    .filters {{

        grid-template-columns:
        1fr;

    }}

    .header h1 {{

        font-size: 25px;

    }}

}}

</style>

</head>


<body>


<div class="container">


<!-- =====================================================
     HEADER
     ===================================================== -->

<div class="header">

<h1>
🚜 Holland Agricultural Machinery
</h1>

<p>
Tractor Showroom & Harvesting Machinery Analytics
</p>

</div>


<!-- =====================================================
     FILTERS
     ===================================================== -->

<div class="filters">


<div class="filter">

<label>
🚜 Category
</label>

<select
id="categoryFilter"
onchange="updateDashboard()">

<option value="All">
All Categories
</option>

</select>

</div>


<div class="filter">

<label>
🏭 Brand
</label>

<select
id="brandFilter"
onchange="updateDashboard()">

<option value="All">
All Brands
</option>

</select>

</div>


<div class="filter">

<label>
📦 Status
</label>

<select
id="statusFilter"
onchange="updateDashboard()">

<option value="All">
All Status
</option>

</select>

</div>


<div class="filter">

<label>
🔎 Search
</label>

<input
id="searchBox"
type="text"
placeholder="Search tractor, brand or model..."
oninput="updateDashboard()">

</div>


</div>


<div class="reset-area">

<button
class="reset-btn"
onclick="resetFilters()">

↻ Reset Filters

</button>

</div>


<!-- =====================================================
     KPI CARDS
     ===================================================== -->

<div class="kpis">


<div class="kpi">

<div class="kpi-title">
TOTAL MACHINES
</div>

<div
class="kpi-value"
id="totalMachines">
0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
INVENTORY VALUE
</div>

<div
class="kpi-value"
id="inventoryValue">
€0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
AVERAGE PRICE
</div>

<div
class="kpi-value"
id="averagePrice">
€0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
AVG WORKING HOURS
</div>

<div
class="kpi-value"
id="averageHours">
0
</div>

</div>


<div class="kpi">

<div class="kpi-title">
IN STOCK
</div>

<div
class="kpi-value"
id="inStock">
0
</div>

</div>


</div>


<!-- =====================================================
     CHARTS
     ===================================================== -->

<div class="chart-grid">


<div class="chart-card">

<div id="categoryChart"></div>

</div>


<div class="chart-card">

<div id="brandChart"></div>

</div>


<div class="chart-card">

<div id="priceChart"></div>

</div>


<div class="chart-card">

<div id="statusChart"></div>

</div>


<div class="chart-card">

<div id="scatterChart"></div>

</div>


<div class="chart-card">

<div id="conditionChart"></div>

</div>


</div>


<!-- =====================================================
     TABLE
     ===================================================== -->

<div class="table-card">

<div class="table-title">

📋 Machinery Inventory

</div>


<div
id="noResults"
class="no-results">

⚠️ No machinery matches your selected filters.

</div>


<div class="table-wrapper">

<table id="dataTable">

<thead>

<tr>

<th>Brand</th>
<th>Model</th>
<th>Category</th>
<th>Year</th>
<th>Price (€)</th>
<th>Working Hours</th>
<th>Condition</th>
<th>Status</th>

</tr>

</thead>

<tbody>
</tbody>

</table>

</div>

</div>


<div class="footer">

🇳🇱 Holland Agricultural Machinery
<br>
Interactive Tractor & Harvesting Machinery Dashboard

</div>


</div>


<script>


// ==========================================================
// DATA
// ==========================================================

const originalData = {data_json};


// ==========================================================
// NORMALIZE TEXT
// ==========================================================
// IMPORTANT FIX
// This makes filtering ignore:
// - spaces
// - capital letters
// - hidden whitespace
// ==========================================================

function normalize(value) {{

    return String(value ?? "")
        .trim()
        .replace(/\\s+/g, " ")
        .toLowerCase();

}}


// ==========================================================
// FILTER INITIALIZATION
// ==========================================================

function initializeFilters() {{

    const categories =
        [...new Set(
            originalData.map(
                row => String(row.Category).trim()
            )
        )]
        .filter(Boolean)
        .sort();


    const brands =
        [...new Set(
            originalData.map(
                row => String(row.Brand).trim()
            )
        )]
        .filter(Boolean)
        .sort();


    const statuses =
        [...new Set(
            originalData.map(
                row => String(row.Status).trim()
            )
        )]
        .filter(Boolean)
        .sort();


    const categorySelect =
        document.getElementById(
            "categoryFilter"
        );


    const brandSelect =
        document.getElementById(
            "brandFilter"
        );


    const statusSelect =
        document.getElementById(
            "statusFilter"
        );


    categories.forEach(
        value => {{

            let option =
                document.createElement(
                    "option"
                );

            option.value = value;

            option.textContent = value;

            categorySelect.appendChild(
                option
            );

        }}
    );


    brands.forEach(
        value => {{

            let option =
                document.createElement(
                    "option"
                );

            option.value = value;

            option.textContent = value;

            brandSelect.appendChild(
                option
            );

        }}
    );


    statuses.forEach(
        value => {{

            let option =
                document.createElement(
                    "option"
                );

            option.value = value;

            option.textContent = value;

            statusSelect.appendChild(
                option
            );

        }}
    );

}}


// ==========================================================
// FILTER DATA
// ==========================================================

function getFilteredData() {{

    const selectedCategory =
        normalize(
            document.getElementById(
                "categoryFilter"
            ).value
        );


    const selectedBrand =
        normalize(
            document.getElementById(
                "brandFilter"
            ).value
        );


    const selectedStatus =
        normalize(
            document.getElementById(
                "statusFilter"
            ).value
        );


    const search =
        normalize(
            document.getElementById(
                "searchBox"
            ).value
        );


    return originalData.filter(
        row => {{

            const category =
                normalize(row.Category);

            const brand =
                normalize(row.Brand);

            const status =
                normalize(row.Status);

            const model =
                normalize(row.Model);

            const categoryMatch =
                selectedCategory === "all" ||
                category === selectedCategory;


            const brandMatch =
                selectedBrand === "all" ||
                brand === selectedBrand;


            const statusMatch =
                selectedStatus === "all" ||
                status === selectedStatus;


            const searchMatch =
                search === "" ||

                brand.includes(search) ||

                model.includes(search) ||

                category.includes(search);


            return (
                categoryMatch &&
                brandMatch &&
                statusMatch &&
                searchMatch
            );

        }}
    );

}}


// ==========================================================
// EURO FORMAT
// ==========================================================

function euro(value) {{

    return new Intl.NumberFormat(
        "en-US",
        {{
            style: "currency",
            currency: "EUR",
            maximumFractionDigits: 0
        }}
    ).format(value || 0);

}}


// ==========================================================
// UPDATE KPIs
// ==========================================================

function updateKPIs(filtered) {{

    const total =
        filtered.length;


    const inventory =
        filtered.reduce(
            (sum,row) =>
            sum +
            Number(row.Price_EUR || 0),
            0
        );


    const averagePrice =
        total
        ? inventory / total
        : 0;


    const averageHours =
        total
        ? filtered.reduce(
            (sum,row) =>
            sum +
            Number(
                row.Working_Hours || 0
            ),
            0
        ) / total
        : 0;


    const stock =
        filtered.filter(
            row =>
            normalize(row.Status)
            === "in stock"
        ).length;


    document.getElementById(
        "totalMachines"
    ).textContent =
        total;


    document.getElementById(
        "inventoryValue"
    ).textContent =
        euro(inventory);


    document.getElementById(
        "averagePrice"
    ).textContent =
        euro(averagePrice);


    document.getElementById(
        "averageHours"
    ).textContent =
        Math.round(
            averageHours
        ).toLocaleString();


    document.getElementById(
        "inStock"
    ).textContent =
        stock;

}}


// ==========================================================
// EMPTY CHART MESSAGE
// ==========================================================

function emptyChart(
    elementId,
    title
) {{

    Plotly.newPlot(
        elementId,
        [],
        {{
            title: {{
                text: title,
                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},
            annotations: [{{
                text:
                "No matching data",
                showarrow: false,
                font: {{
                    size: 16,
                    color: "#C76D4F"
                }}
            }}],
            paper_bgcolor: "white",
            plot_bgcolor: "white"
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// CATEGORY CHART
// ==========================================================

function categoryChart(filtered) {{

    if (!filtered.length) {{
        emptyChart(
            "categoryChart",
            "🚜 Machines by Category"
        );
        return;
    }}


    const counts = {{}};


    filtered.forEach(
        row => {{

            const key =
                row.Category;

            counts[key] =
                (counts[key] || 0)
                + 1;

        }}
    );


    const entries =
        Object.entries(counts)
        .sort(
            (a,b) => a[1] - b[1]
        );


    Plotly.newPlot(
        "categoryChart",
        [{{
            x:
                entries.map(x => x[1]),

            y:
                entries.map(x => x[0]),

            type: "bar",

            orientation: "h",

            marker: {{
                color: "#2E8B57"
            }}
        }}],
        {{
            title: {{
                text:
                "🚜 Machines by Category",

                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},

            paper_bgcolor: "white",

            plot_bgcolor: "white",

            margin: {{
                l: 130,
                r: 30,
                t: 60,
                b: 50
            }}
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// BRAND CHART
// ==========================================================

function brandChart(filtered) {{

    if (!filtered.length) {{
        emptyChart(
            "brandChart",
            "🏭 Top Machinery Brands"
        );
        return;
    }}


    const counts = {{}};


    filtered.forEach(
        row => {{

            const key =
                row.Brand;

            counts[key] =
                (counts[key] || 0)
                + 1;

        }}
    );


    const entries =
        Object.entries(counts)
        .sort(
            (a,b) => b[1] - a[1]
        )
        .slice(0,10);


    Plotly.newPlot(
        "brandChart",
        [{{
            x:
                entries.map(x => x[1]),

            y:
                entries.map(x => x[0]),

            type: "bar",

            orientation: "h",

            marker: {{
                color: "#4F8A7A"
            }}
        }}],
        {{
            title: {{
                text:
                "🏭 Top Machinery Brands",

                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},

            paper_bgcolor: "white",

            plot_bgcolor: "white",

            margin: {{
                l: 130,
                r: 30,
                t: 60,
                b: 50
            }}
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// PRICE CHART
// ==========================================================

function priceChart(filtered) {{

    if (!filtered.length) {{
        emptyChart(
            "priceChart",
            "💰 Average Price by Category"
        );
        return;
    }}


    const groups = {{}};


    filtered.forEach(
        row => {{

            if (!groups[row.Category]) {{
                groups[row.Category] = [];
            }}

            groups[row.Category].push(
                Number(row.Price_EUR || 0)
            );

        }}
    );


    const categories =
        Object.keys(groups);


    const averages =
        categories.map(
            category => {{

                const values =
                    groups[category];

                return values.reduce(
                    (a,b) => a+b,
                    0
                ) / values.length;

            }}
        );


    Plotly.newPlot(
        "priceChart",
        [{{
            x: categories,

            y: averages,

            type: "bar",

            marker: {{
                color: "#D4A72C"
            }}
        }}],
        {{
            title: {{
                text:
                "💰 Average Price by Category",

                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},

            paper_bgcolor: "white",

            plot_bgcolor: "white",

            yaxis: {{
                title:
                "Average Price (€)"
            }}
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// STATUS CHART
// ==========================================================

function statusChart(filtered) {{

    if (!filtered.length) {{
        emptyChart(
            "statusChart",
            "📦 Inventory Status"
        );
        return;
    }}


    const counts = {{}};


    filtered.forEach(
        row => {{

            const key =
                row.Status;

            counts[key] =
                (counts[key] || 0)
                + 1;

        }}
    );


    Plotly.newPlot(
        "statusChart",
        [{{
            labels:
                Object.keys(counts),

            values:
                Object.values(counts),

            type: "pie",

            hole: .55,

            marker: {{
                colors: [
                    "#2E8B57",
                    "#D4A72C",
                    "#4F8A7A",
                    "#C76D4F"
                ]
            }}
        }}],
        {{
            title: {{
                text:
                "📦 Inventory Status",

                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},

            paper_bgcolor:
            "white"
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// SCATTER CHART
// ==========================================================

function scatterChart(filtered) {{

    if (!filtered.length) {{
        emptyChart(
            "scatterChart",
            "📈 Price vs Working Hours"
        );
        return;
    }}


    const groups = {{}};


    filtered.forEach(
        row => {{

            const category =
                row.Category;


            if (!groups[category]) {{

                groups[category] = {{
                    x: [],
                    y: [],
                    text: []
                }};

            }}


            groups[category].x.push(
                Number(
                    row.Working_Hours || 0
                )
            );


            groups[category].y.push(
                Number(
                    row.Price_EUR || 0
                )
            );


            groups[category].text.push(
                row.Brand +
                " " +
                row.Model
            );

        }}
    );


    const traces =
        Object.keys(groups)
        .map(
            category => ({{

                x:
                    groups[category].x,

                y:
                    groups[category].y,

                text:
                    groups[category].text,

                mode:
                    "markers",

                type:
                    "scatter",

                name:
                    category,

                marker: {{
                    size: 10
                }}

            }})
        );


    Plotly.newPlot(
        "scatterChart",
        traces,
        {{
            title: {{
                text:
                "📈 Price vs Working Hours",

                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},

            xaxis: {{
                title:
                "Working Hours"
            }},

            yaxis: {{
                title:
                "Price (€)"
            }},

            paper_bgcolor:
            "white",

            plot_bgcolor:
            "white"
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// CONDITION CHART
// ==========================================================

function conditionChart(filtered) {{

    if (!filtered.length) {{
        emptyChart(
            "conditionChart",
            "🔧 Average Price by Condition"
        );
        return;
    }}


    const groups = {{}};


    filtered.forEach(
        row => {{

            const condition =
                row.Condition ||
                "Unknown";


            if (!groups[condition]) {{
                groups[condition] = [];
            }}


            groups[condition].push(
                Number(
                    row.Price_EUR || 0
                )
            );

        }}
    );


    const labels =
        Object.keys(groups);


    const values =
        labels.map(
            condition => {{

                const arr =
                    groups[condition];

                return arr.reduce(
                    (a,b) => a+b,
                    0
                ) / arr.length;

            }}
        );


    Plotly.newPlot(
        "conditionChart",
        [{{
            x: labels,

            y: values,

            type: "bar",

            marker: {{
                color: "#A8C3B0"
            }}
        }}],
        {{
            title: {{
                text:
                "🔧 Average Price by Condition",

                font: {{
                    size: 20,
                    color: "#12372A"
                }}
            }},

            yaxis: {{
                title:
                "Average Price (€)"
            }},

            paper_bgcolor:
            "white",

            plot_bgcolor:
            "white"
        }},
        {{
            responsive: true
        }}
    );

}}


// ==========================================================
// TABLE
// ==========================================================

function updateTable(filtered) {{

    const tbody =
        document.querySelector(
            "#dataTable tbody"
        );


    const noResults =
        document.getElementById(
            "noResults"
        );


    tbody.innerHTML = "";


    if (!filtered.length) {{

        noResults.style.display =
            "block";

        return;

    }} else {{

        noResults.style.display =
            "none";

    }}


    filtered.forEach(
        row => {{

            const tr =
                document.createElement(
                    "tr"
                );


            tr.innerHTML = `

                <td>
                    ${{row.Brand || ""}}
                </td>

                <td>
                    ${{row.Model || ""}}
                </td>

                <td>
                    ${{row.Category || ""}}
                </td>

                <td>
                    ${{row.Year || ""}}
                </td>

                <td>
                    ${{euro(
                        Number(
                            row.Price_EUR || 0
                        )
                    )}}
                </td>

                <td>
                    ${{
                        Number(
                            row.Working_Hours || 0
                        ).toLocaleString()
                    }}
                </td>

                <td>
                    ${{row.Condition || ""}}
                </td>

                <td>
                    ${{row.Status || ""}}
                </td>

            `;


            tbody.appendChild(
                tr
            );

        }}
    );

}}


// ==========================================================
// MAIN UPDATE
// ==========================================================

function updateDashboard() {{

    const filtered =
        getFilteredData();


    updateKPIs(
        filtered
    );


    categoryChart(
        filtered
    );


    brandChart(
        filtered
    );


    priceChart(
        filtered
    );


    statusChart(
        filtered
    );


    scatterChart(
        filtered
    );


    conditionChart(
        filtered
    );


    updateTable(
        filtered
    );

}}


// ==========================================================
// RESET
// ==========================================================

function resetFilters() {{

    document.getElementById(
        "categoryFilter"
    ).value = "All";


    document.getElementById(
        "brandFilter"
    ).value = "All";


    document.getElementById(
        "statusFilter"
    ).value = "All";


    document.getElementById(
        "searchBox"
    ).value = "";


    updateDashboard();

}}


// ==========================================================
// START
// ==========================================================

initializeFilters();

updateDashboard();


</script>

</body>

</html>
"""


# ============================================================
# 13. SAVE HTML
# ============================================================

dashboard_file = os.path.join(
    OUTPUT_DIR,
    "holland_agricultural_machinery_dashboard.html"
)

with open(
    dashboard_file,
    "w",
    encoding="utf-8"
) as file:

    file.write(html)


# ============================================================
# 14. FINAL INFORMATION
# ============================================================

print("\n" + "=" * 70)

print("🎉 DASHBOARD CREATED SUCCESSFULLY!")

print("=" * 70)

print("\n🚜 Dashboard:")
print(dashboard_file)

print("\n📊 Total Machines:", total_machines)

print(
    "💰 Inventory Value: €{:,.0f}".format(
        inventory_value
    )
)

print(
    "💵 Average Price: €{:,.0f}".format(
        average_price
    )
)

print(
    "⏱️ Average Working Hours: {:,.0f}".format(
        average_hours
    )
)

print(
    "📦 In Stock:",
    in_stock
)

print(
    "💼 Sold:",
    sold
)

print(
    "🔒 Reserved:",
    reserved
)

print(
    "🚜 Available Soon:",
    available_soon
)

print("\n📁 Output folder:")
print(OUTPUT_DIR)

print("\n🌐 Opening dashboard...")

print("=" * 70)


# ============================================================
# 15. OPEN IN BROWSER
# ============================================================

webbrowser.open(
    Path(dashboard_file)
    .resolve()
    .as_uri()
)

print("\n✅ DONE!")

✅ Required Python libraries installed!

🇳🇱 HOLLAND AGRICULTURAL MACHINERY
✅ Dataset loaded successfully!
Records: 100
✅ All required columns found!

🎉 DASHBOARD CREATED SUCCESSFULLY!

🚜 Dashboard:
C:\Users\Nasar\Documents\DATAANALYSIS\tractor_analysis_output\holland_agricultural_machinery_dashboard.html

📊 Total Machines: 100
💰 Inventory Value: €16,759,030
💵 Average Price: €167,590
⏱️ Average Working Hours: 4,382
📦 In Stock: 26
💼 Sold: 31
🔒 Reserved: 25
🚜 Available Soon: 18

📁 Output folder:
C:\Users\Nasar\Documents\DATAANALYSIS\tractor_analysis_output

🌐 Opening dashboard...

✅ DONE!


C:\Users\Nasar\AppData\Local\Temp\ipykernel_12156\2109952777.py:83: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
